## Fetch trades and quotes for stock and one option

In [ ]:
import os
#os.environ["QUANTDATA_CACHE_MODE"] = "off" # turns off global cache mode ... good for testing but not in general

%load_ext autoreload
%autoreload 1

%aimport quantdata
%aimport quantdata.biz_calendar
%aimport quantdata.massive.client
%aimport quantdata.massive.options.contracts
%aimport quantdata.massive.processing

from datetime import date, datetime
import warnings
from quantdata.massive.client import MassiveClient
from cdxcore.util import fmt_date, fmt_dict
from cdxcore.verbose import Context
from quantdata.massive.processing import extract_trades

import polars as pl

def test():

    ticker = "SPY"
    d      = date(2024,1,3)
    client = MassiveClient()
    option_ticker = "O:SPY240105P00447000"

    def package( ticker, verbose = Context.all ):

        with verbose.write_t(f"Fetching OCV, trades, quotes for {ticker} on {fmt_date(d)}... ", end='') as tme:
            ocv = client.fetch_day_summary(ticker, d, adjusted=True)
            verbose.write(f"done: found {fmt_dict(ocv)}. This took {tme}.", head=False)

        with verbose.write_t(f"Fetching trades for {ticker} on {fmt_date(d)}... ", end='') as tme:
            trades = extract_trades(ticker, d) # extracts trades, removed zero size and price, and excludes trades with NORMAL_TRADE_EXCLUDE_FLAGS
            verbose.write(f"done: found {len(trades)} trades. This took {tme}.", head=False)

        with verbose.write_t(f"Fetching quotes for {ticker} on {fmt_date(d)}... ", end='') as tme:
            quotes = client.fetch_quotes(ticker, d)
            verbose.write(f"done: found {len(quotes)} quotes. This took {tme}.", head=False)



        return ocv, trades, quotes  

    package(ticker)
    ocv, trades, quotes = package(option_ticker)

    display(ocv)
    display(trades.head())
    display(quotes.head())

test()



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
00: Fetching OCV, trades, quotes for SPY on 2024-01-03... done: found open: 470.43, close: 468.79, volume: 103585866.0, high: 471.19, low: 468.17, afterHours: 469.35, preMarket: 472.07 and adjusted: True. This took 0.56s.
00: Fetching trades for SPY on 2024-01-03... done: found 194426 trades. This took 1:48.
00: Fetching quotes for SPY on 2024-01-03... 

/mnt/c/Users/hans/OneDrive/Python3/quant_data/quantdata/biz_calendar.py:433: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
/mnt/c/Users/hans/OneDrive/Python3/quant_data/quantdata/massive/client.py:201: UserWarning: **** Massive error: Response ended prematurely ...sleeping for 1 seconds ***


                         market_open              market_close
2024-12-24 2024-12-24 14:30:00+00:00 2024-12-24 18:00:00+00:00
X 2024-12-24 09:30:00-05:00 2024-12-24 13:00:00-05:00


/tmp/ipykernel_106775/3200204046.py:19: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
